

# APTOS: Three Predictions I Wrote Before the Data

A diabetic retinopathy grader on APTOS-2019 reaches a five-fold ensemble QWK of **0.909**. Public solutions reach 0.93, so the score is not why this notebook exists.

What is worth showing is what happened when every claim about that model was tested, and each prediction was committed to GitHub **before** the data it concerned was touched:

1. A shortcut exists: file metadata alone predicts the grade. **Recomputed live below on this dataset.**
2. On **IDRiD**, where that simple resolution proxy is constant, I predicted the model would collapse if it depended on it. It did not - the prediction failed in the model's favour.
3. On **Messidor-2**, graded by a panel of retina specialists, I predicted referable AUC of at least 0.93. It came out **0.819** - the prediction failed against the model.
4. A fine-tuning experiment with a **control arm** supports a role for grading labels, without ruling out all acquisition effects.

Full project: [GitHub](https://github.com/senanurcetin/APTOS-2019-diabetic-retinopathy) | [live demo](https://aptos-2019-diabetic-retinopathy.onrender.com) | [model card](https://huggingface.co/senanurcetin/aptos-retinopathy-grader) | [write-up](https://github.com/senanurcetin/APTOS-2019-diabetic-retinopathy/blob/main/docs/WRITEUP.md)

The images are the training set of the [APTOS 2019 Blindness Detection](https://www.kaggle.com/competitions/aptos2019-blindness-detection) competition, used here through a copy pre-split into train, validation and test ([mariaherrerot/aptos2019](https://www.kaggle.com/datasets/mariaherrerot/aptos2019)).

*A research project, not a medical device.*

**NEXUS / OPEN RESEARCH.** Noncommercial historical research; not a medical device. Recorded external results are read from the release; the metadata classifier is recomputed here. No new imaging model is trained.


In [ ]:
import glob
import io
import json
import os
import urllib.request

import cv2
import matplotlib.pyplot as plt
import pandas as pd
from joblib import Parallel, delayed
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import cohen_kappa_score
from sklearn.model_selection import StratifiedKFold, cross_val_predict

# Results of the GPU experiments are read from the repository at the v1.0.0
# release tag, so every number below matches the published reports exactly.
RELEASE = "https://raw.githubusercontent.com/senanurcetin/APTOS-2019-diabetic-retinopathy/v1.0.0/reports/"

def report(name):
    with urllib.request.urlopen(RELEASE + name, timeout=30) as response:
        data = response.read().decode("utf-8")
    return json.loads(data) if name.endswith(".json") else pd.read_csv(io.StringIO(data))

calibration = report("calibration.json")
finetune = report("finetune_messidor2.json")
messidor = report("external_validation_messidor2.json")
runs = report("runs.csv")
print("reports loaded from the v1.0.0 release")
plt.rcParams.update({"figure.facecolor":"#0B0F19", "axes.facecolor":"#0B0F19", "text.color":"#B9C5CE", "axes.labelcolor":"#B9C5CE", "xtick.color":"#B9C5CE", "ytick.color":"#B9C5CE", "axes.edgecolor":"#4EA8FF"})


## The whole story in one chart

Referable DR (grade 2 or above) is the screening decision, and ROC AUC measures how well the model ranks eyes for it, independent of any threshold.

In [ ]:
applied = {row["set"]: row for row in calibration["applied"]}
bars = [
    ("APTOS test\n(in-distribution)", applied["APTOS test"]["ROC AUC"], "#4EA8FF"),
    ("IDRiD\n(external)", applied["IDRiD"]["ROC AUC"], "#4EA8FF"),
    ("Messidor-2\n(external)", applied["Messidor-2"]["ROC AUC"], "#8F9BFF"),
    ("Messidor-2 held-out half\nafter fine-tuning\non specialist labels", finetune["adjudicated"]["Messidor-2 test half"]["referable_auc"], "#00E0FF"),
    ("Messidor-2 held-out half\ncontrol: same images,\nmodel's own labels", finetune["control"]["Messidor-2 test half"]["referable_auc"], "#8c8c8c"),
]
fig, ax = plt.subplots(figsize=(11, 5.2))
labels, values, colours = zip(*bars, strict=True)
positions = range(len(bars))
ax.bar(positions, values, color=colours, width=0.62)
for x, v in zip(positions, values, strict=True):
    ax.text(x, v + 0.005, f"{v:.3f}", ha="center", va="bottom", fontsize=12, fontweight="bold")
ax.axhline(0.93, ls="--", color="#333333", lw=1)
ax.text(1.62, 0.947, "pre-registered prediction for Messidor-2: AUC >= 0.93", ha="left", va="bottom", fontsize=9, color="#333333")
ax.set_xticks(list(positions))
ax.set_xticklabels(labels, fontsize=9.5)
ax.set_ylim(0.75, 1.0)
ax.set_ylabel("referable ROC AUC")
ax.set_title("A retinopathy model that reads the retina - through its training labels", fontsize=13, loc="left")
for side in ("top", "right"):
    ax.spines[side].set_visible(False)
plt.tight_layout()
plt.show()

## 1. The shortcut, recomputed on this dataset

APTOS was collected at several sites with different cameras, and the camera correlates with disease. If that is true, a model using only global file/image summaries should still predict the grade.

The cell below reads every image in this Kaggle dataset, extracts **only file properties** - width, height, aspect ratio, megapixels, file size, mean brightness and contrast - and cross-validates a random forest on them. Nothing here is loaded from the repository.

In [ ]:
# The mirror below splits this competition's 3,662 training images into
# train/valid/test. Search two levels deep only: the competition data is also
# mounted, and a recursive walk would visit every one of its images.
roots = (glob.glob("/kaggle/input/*/train_1.csv") + glob.glob("/kaggle/input/*/*/train_1.csv")
         or glob.glob("/kaggle/input/**/train_1.csv", recursive=True))
ROOT = os.path.dirname(roots[0]) if roots else os.environ.get("APTOS_DATA", "data/images")

folders = {"train_1.csv": "train_images/train_images",
           "valid.csv": "val_images/val_images",
           "test.csv": "test_images/test_images"}
labels = pd.concat([
    pd.read_csv(os.path.join(ROOT, name)).dropna(subset=["id_code"]).assign(folder=folder)
    for name, folder in folders.items()
], ignore_index=True)
labels["diagnosis"] = labels["diagnosis"].astype(int)
print(f"{len(labels)} labelled images")

def metadata(row):
    path = os.path.join(ROOT, row.folder, f"{row.id_code}.png")
    gray = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    h, w = gray.shape
    return {"id_code": row.id_code, "width": w, "height": h,
            "aspect_ratio": w / h, "megapixels": w * h / 1e6,
            "file_kb": os.path.getsize(path) / 1024,
            "brightness": float(gray.mean()), "contrast_std": float(gray.std())}

meta = pd.DataFrame(Parallel(n_jobs=-1)(delayed(metadata)(row) for row in labels.itertuples()))
data = labels.merge(meta, on="id_code")
print(f"{data[['width', 'height']].drop_duplicates().shape[0]} distinct resolutions")

In [ ]:
features = ["width", "height", "aspect_ratio", "megapixels", "file_kb", "brightness", "contrast_std"]
folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
pred = cross_val_predict(RandomForestClassifier(n_estimators=200, random_state=0, n_jobs=-1),
                         data[features], data["diagnosis"], cv=folds)
shortcut_qwk = cohen_kappa_score(data["diagnosis"], pred, weights="quadratic")
print(f"QWK from file metadata alone, 5-fold CV: {shortcut_qwk:.3f}")
print("QWK from always predicting 'No DR':      0.000")
print("(the repository reports 0.652 with its own fold split)")

In [ ]:
square = (data["width"] == 1050) & (data["height"] == 1050)
table = pd.DataFrame({
    "images": [square.sum(), (~square).sum()],
    "share graded No DR": [(data.loc[square, "diagnosis"] == 0).mean(),
                           (data.loc[~square, "diagnosis"] == 0).mean()],
}, index=["1050x1050 (most common resolution)", "every other resolution"])
table.style.format({"share graded No DR": "{:.1%}"})

Most images at the most common resolution are healthy; elsewhere the mix is very different. Resizing every image to 512px does not remove this, because aspect ratio and sharpness survive it. **The metadata baseline is evidence of a shortcut, not a mathematical lower bound on an image model** - and every later question becomes: how much of the score is the retina?

## 2. Cross-validation, and two ideas that did nothing

The model is an EfficientNet-B0 ordinal regressor with validation-fitted thresholds, trained in five folds. Two preprocessing ideas with good arguments behind them - CLAHE contrast enhancement, and squashing images to square instead of padding - were compared on paired folds:

In [ ]:
cv = runs[(runs["split"] == "cv-fold") & (runs["stratify"] == "diagnosis")]
per_fold = cv.pivot_table(index="fold", columns="variant", values="test_qwk")[["baseline", "squash", "clahe"]]
summary = runs[(runs["split"] == "cv-summary") & (runs["stratify"] == "diagnosis")].set_index("variant")
pd.DataFrame({
    "mean test QWK per fold": per_fold.mean(),
    "five-fold ensemble QWK": summary["ensemble_qwk"],
}).round(4)

Both differences are within fold-to-fold noise (paired t-test p = 0.47 for squash, p = 0.87 for CLAHE). The largest effect in the whole project is ensembling the five folds: about +0.019 QWK.

## 3. External test 1 - IDRiD: a prediction that failed in the model's favour

IDRiD (455 images from another Indian clinic) was captured at **one resolution**, so image resolution alone carries no label information there. Better still, it inverts it: in APTOS every image at IDRiD's resolution is diseased; in IDRiD 28% are healthy.

**Written before the run:** if the model leans on acquisition cues, it will call IDRiD's healthy eyes diseased and specificity will collapse.

In [ ]:
rows = ["APTOS test", "IDRiD"]
pd.DataFrame({name: {"referable ROC AUC": applied[name]["ROC AUC"],
                     "sensitivity at the APTOS cut": applied[name]["sensitivity"],
                     "specificity at the APTOS cut": applied[name]["specificity"]}
              for name in rows}).round(3)

Specificity went *up*. On the dataset built to expose the shortcut, the model ranked patients as well as at home. **This rejects exclusive reliance on the measured simple geometry proxy, but does not exclude other acquisition confounds.** What drifted was calibration: a cut chosen for 90% sensitivity on APTOS gave 82% on IDRiD.

## 4. External test 2 - Messidor-2: a prediction that failed against the model

One external set can be luck. Messidor-2 (France, 1744 images) adds something new: every image was graded by a **panel of three retina specialists** who adjudicated disagreements (Krause et al., 2018). APTOS labels come from single graders, and duplicate images in APTOS disagree with themselves 29% of the time.

**Written before the run:** referable AUC >= 0.93.

In [ ]:
print(f"Messidor-2 referable ROC AUC: {applied['Messidor-2']['ROC AUC']:.3f}")
print(f"Sensitivity at the APTOS cut: {applied['Messidor-2']['sensitivity']:.3f}")
recall = pd.Series(messidor["per_class_recall"]).astype(float)
recall.index = ["0 No DR", "1 Mild", "2 Moderate", "3 Severe", "4 Proliferative"]
recall.round(3).to_frame("recall on Messidor-2")

The prediction failed. The loss sits at **grade 2, Moderate** - the referral boundary: most Moderate eyes are graded below 2. In the repository's analysis, Moderate eyes with hard exudates were caught, and those defined by subtler signs were scored like Mild.

The reading after the fact: the model learned where APTOS's single graders draw the Mild/Moderate line, and the specialist panel draws it lower. That is a hypothesis, so it got its own experiment.

## 5. Testing the cause, with a control arm

Messidor-2 was split using **filename-derived proxy groups** (exam dates and consecutive IM-number chains). These groups do not overlap across partitions, but authentic patient linkage was unavailable: separation of every patient's two eyes cannot be verified. The five fold models were fine-tuned on one partition and evaluated on the other.

The obvious objection: fine-tuning also shows the model Messidor-2's cameras, and that alone might help. So a **control** arm was fine-tuned on the same images with the original model's own grades as labels - new cameras, old boundary.

In [ ]:
arms = {"unchanged": "none", "specialist labels": "adjudicated", "control (model's own labels)": "control"}
pd.DataFrame({
    label: {"Messidor-2 held-out AUC": finetune[key]["Messidor-2 test half"]["referable_auc"],
            "Moderate eyes graded below 2": finetune[key]["Messidor-2 test half"]["moderate_below_2"],
            "IDRiD AUC": finetune[key]["IDRiD"]["referable_auc"],
            "APTOS test AUC": finetune[key]["APTOS test"]["referable_auc"],
            "APTOS test accuracy": finetune[key]["APTOS test"]["accuracy"]}
    for label, key in arms.items()
}).round(3)

The control gains nothing; the specialist labels move the boundary. **The controlled comparison supports a role for grading labels; it does not rule out all acquisition effects.**

Two things did not go the way the hypothesis said, and both are reported in the repository:
- The whole Moderate grade shifted up, rather than the subtle cases specifically - the model redrew a line; there is no sign it learned to see anything new.
- The fine-tuned model over-grades APTOS (accuracy falls), so it stays an experiment and the deployed model is unchanged.

## What I take from it

- **A benchmark score is a claim, not a result.** The metadata baseline achieved QWK around 0.65 without retinal structure as an explicit input.
- **External data answers questions internal splits cannot.** IDRiD tested one simple geometry shortcut; other confounds remain possible.
- **Models inherit their labels.** Label quality matters; these experiments do not identify a formal accuracy ceiling or exclude architectural limits.
- **Writing the prediction down first** is what made the failures informative instead of embarrassing.

Everything - code, the pre-registered predictions with their commit timestamps, a versioned test suite, and the serving stack - is in the [repository](https://github.com/senanurcetin/APTOS-2019-diabetic-retinopathy). You can try the deployed model in the [live demo](https://aptos-2019-diabetic-retinopathy.onrender.com).

*Data: APTOS 2019 Blindness Detection; IDRiD; Messidor-2 with adjudicated grades published by Google (Krause et al., 2018). Not a medical device.*

## Two thresholds must not be mixed

The original grade-2 boundary (~1.132) gives sensitivity 88.5% on IDRiD and 32.8% on Messidor-2. The separate APTOS-selected operating point (1.3359375), used by the calibration tables above, gives 81.6% and 26.0%. Both are frozen before external application. Wilson intervals are descriptive; aggregate files cannot adjust for bilateral patient clusters.

Five fold models are evaluated on one shared APTOS test set, so their SD is not an outer-CV interval. The duplicate-pair analysis does not establish a formal accuracy ceiling. No external result establishes prospective clinical safety.


## Methodological review and a separate completed GPU run

The 24 October 2026 [scientific assessment](https://github.com/senanurcetin/APTOS-2019-diabetic-retinopathy/blob/main/docs/SCIENTIFIC_REVIEW.md) distinguishes filename-proxy groups from authenticated patient linkage, exploratory repeated-holdout comparisons from confirmatory evaluation, and global brightness/contrast summaries from spatial retinal features.

A separate scalar MSE seed-42 sweep (`baseline-c8232af68337`) completed five folds. Its saved test predictions reproduce ensemble QWK **0.900763**, accuracy **0.808743**, and macro F1 **0.551072**. This result is separate from the historical external-evaluation baseline above. It does not establish an ordinal-model improvement or cross-site transfer. Checkpoint inference was not rerun by the audit. [Aggregate audit and source hashes](https://github.com/senanurcetin/APTOS-2019-diabetic-retinopathy/blob/main/reports/research/kaggle_reg42_audit.json).
